# Character N-Grams

En este notebook usamos TF-IDF basado en secuencias de caracteres en lugar de palabras. Los n-grams de caracteres capturan patrones ortográficos y morfológicos que evolucionaron a lo largo del tiempo: grafías arcaicas, terminaciones verbales y el uso de letras que cambiaron entre los siglos XVI y XIX. Esto hace que este enfoque sea especialmente adecuado para detectar la época de un texto.

## 1. Importación de librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay

## 2. Carga de los datos

In [ ]:
df = pd.read_csv('train.csv')
df_eval = pd.read_csv('eval.csv')

In [ ]:
df.head()

In [ ]:
df.shape

## 3. Exploración del conjunto de datos

In [ ]:
df['decade'].value_counts().sort_index().plot(
    kind='bar', figsize=(14, 4), title='Distribución de décadas'
)
plt.xlabel('Década')
plt.ylabel('Frecuencia')
plt.tight_layout()
plt.show()

In [ ]:
def reporte_calidad(df):
    reporte_de_cualidad = {
        'Total records': len(df),
        'duplicated record': df.duplicated().sum(),
        'missing values': df.isnull().sum().to_dict(),
        'data types': df.dtypes.astype(str).to_dict(),
    }
    return reporte_de_cualidad

print(reporte_calidad(df))

## 4. Preprocesamiento del texto

Para los n-grams de caracteres aplicamos una limpieza más suave: solo convertimos a minúsculas y normalizamos espacios, pero conservamos la puntuación y otros caracteres que pueden ser informativos desde el punto de vista ortográfico. Eliminar demasiada información haría que este enfoque perdiera su ventaja sobre el basado en palabras.

In [ ]:
def limpiar_texto_char(texto):
    texto = str(texto).lower()
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)
df['texto_limpio'] = df['text'].apply(limpiar_texto_char)
df[['texto_limpio', 'decade']].head()

## 5. Partición de los datos

In [ ]:
X = df['texto_limpio']
y = df['decade']

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=1, stratify=y
)

Usamos `stratify=y` para conservar la proporción de clases en ambas particiones.

In [ ]:
X_train.shape, X_val.shape

## 6. Construcción del pipeline

Usamos `TfidfVectorizer` con `analyzer='char_wb'`, que extrae n-grams de caracteres respetando los límites de palabra. Esta opción es preferible a `'char'` puro porque evita que los n-grams crucen espacios entre palabras de forma arbitraria.

In [ ]:
pipeline_char = Pipeline([
    ('tfidf', TfidfVectorizer(analyzer='char_wb')),
    ('clf', LogisticRegression(max_iter=1000, solver='saga')),
])

## 7. Entrenamiento con búsqueda de hiperparámetros

Exploramos distintos rangos de longitud de n-grams, tamaño de vocabulario y frecuencia mínima de aparición.

In [ ]:
param_grid = {
    'tfidf__ngram_range': [(2, 4), (3, 5), (3, 6)],
    'tfidf__max_features': [30000, 50000],
    'tfidf__min_df': [2, 3],
    'clf__C': [0.1, 1, 10],
}

In [ ]:
kfold = KFold(n_splits=5, shuffle=True, random_state=0)
grid_char = GridSearchCV(
    pipeline_char, param_grid, cv=kfold, scoring='accuracy', n_jobs=-1
)

In [ ]:
grid_char.fit(X_train, y_train)

In [ ]:
print('Mejores hiperparámetros:', grid_char.best_params_)
print('Mejor score CV (accuracy):', round(grid_char.best_score_, 4))

## 8. Evaluación del mejor modelo

In [ ]:
best_model = grid_char.best_estimator_

y_pred_train = best_model.predict(X_train)
y_pred_val   = best_model.predict(X_val)

#### Comparación de rendimientos sobre entrenamiento y validación

In [ ]:
print('Accuracy en entrenamiento:', round(accuracy_score(y_train, y_pred_train), 4))
print('Accuracy en validación:   ', round(accuracy_score(y_val,   y_pred_val),   4))
print('Mejor score CV (accuracy):', round(grid_char.best_score_,                 4))

In [ ]:
print(classification_report(y_val, y_pred_val))

In [ ]:
fig, ax = plt.subplots(figsize=(16, 12))
ConfusionMatrixDisplay.from_predictions(y_val, y_pred_val, ax=ax, colorbar=False)
plt.title('Matriz de confusión — validación')
plt.tight_layout()
plt.show()

## 9. Predicciones sobre eval.csv

In [ ]:
df_eval['texto_limpio'] = df_eval['text'].apply(limpiar_texto_char)

y_eval_pred = best_model.predict(df_eval['texto_limpio'])

submission = pd.DataFrame({'id': df_eval['id'], 'answer': y_eval_pred})
submission.to_csv('submission_char_ngrams.csv', index=False)
submission.head()